## 5、Tool 拓展与注意事项

对应《尚硅谷-05-Tools.pdf》第 5 节「强制使用工具」（PDF 第 35–41 页），
后面补充第 6 节中容易误解的错误处理、返回值和异步调用。

沿用前面笔记的短标题、分块代码和中文注释。选择 `langchain` 内核，按顺序执行。
**离线示例只检查参数和消息流转，不会向模型发请求；最后两个单元才需要真实 API。**
所有天气、新闻和用户数据均为模拟数据。

### 5.1 tool_choice 控制什么？

它控制模型这一次回答是否要提出工具调用请求，以及要选择哪个工具。
**它不会自动执行 Python 函数，也不保证工具查询结果正确。**

| 取值 | 含义／适用情况 |
| --- | --- |
| `"none"` | 本轮不调用工具，例如拿到工具结果后，只生成最终回答 |
| `"auto"` | 让模型自行决定，可能不调用，也可能提出一个或多个调用 |
| `"required"` | 至少调用一个工具；不表示所有工具都要调用 |
| `"any"` | 在本工程的 ChatOpenAI 适配器中，会转换成 `"required"` |
| 工具名，例如 `"get_weather2"` | 指定模型选择已绑定的这个工具 |
| 指定工具的字典 | 更明确地表达要调用哪个函数工具 |

上表要以实际服务端支持为前提。下面先用本地适配器观察这些值如何转换，
再说明你的百炼配置需要注意的兼容性。

In [1]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import ToolException
from pydantic import BaseModel, Field, ValidationError
from typing import Literal
from rich import print as rprint
import json

# 只检查绑定参数：使用占位密钥和 .invalid 地址，不调用 invoke
offline_model = ChatOpenAI(
    model="qwen3.8-max",
    api_key="offline-demo-key",
    base_url="https://example.invalid/v1",
)

@tool(parse_docstring=True)
def get_weather1(city: str) -> str:
    """获取当日天气。

    Args:
        city: 城市名称。
    """
    return f"{city}当天晴朗（模拟来源1）"

@tool(parse_docstring=True)
def get_weather2(city: str) -> str:
    """获取当日天气。

    Args:
        city: 城市名称。
    """
    return f"{city}当天晴朗（模拟来源2）"

@tool
def get_news() -> str:
    """获取当日新闻。"""
    return "模拟新闻：这是一条用于演示工具调用的示例消息。"

tools = [get_weather1, get_weather2, get_news]

#### 先查看绑定结果，不请求模型

`.kwargs` 是绑定给后续模型请求的参数。这里看到的是**本地转换结果**，
不能据此断言百炼或其他服务端一定接受这个参数。

In [2]:
for choice in ["none", "auto", "required", "any", "get_weather2"]:
    bound = offline_model.bind_tools(tools, tool_choice=choice)
    print(f"{choice!r} -> {bound.kwargs['tool_choice']}")

'none' -> none
'auto' -> auto
'required' -> required
'any' -> required
'get_weather2' -> {'type': 'function', 'function': {'name': 'get_weather2'}}


### 5.2 none、False、None 不是一回事

`"none"` 明确要求不调用工具。
在当前 `langchain-openai==1.1.11` 中，Python 的 `False` 和 `None` 都不会显式传入 `tool_choice`；
`True` 会转换成 `"required"`。

课件把默认情况按 `auto` 讲解；本地适配器省略参数后，最终默认行为由服务端决定。
即使设置 `"none"`，模型也不因此具备实时天气知识，不应把它的文字回答当成已查询结果。

In [3]:
for choice in ["none", False, None, True]:
    bound = offline_model.bind_tools(tools, tool_choice=choice)
    print(repr(choice), "->", bound.kwargs.get("tool_choice", "<未显式指定>"))

default_bound = offline_model.bind_tools(tools)
print("不传参数 ->", default_bound.kwargs.get("tool_choice", "<未显式指定>"))

'none' -> none
False -> <未显式指定>
None -> <未显式指定>
True -> required
不传参数 -> <未显式指定>


### 5.3 auto：提供工具，不等于每次都会用工具

「杭州天气如何？」适合借助天气工具；「你好啊」通常不需要。
这只是选择原则，不能把课件里的某次响应当作每次都相同的固定输出。

是否调用、调用哪些工具，要看 `response.tool_calls`；不能只看 `response.content` 是否为空。
带有工具请求的 `AIMessage` 也可能同时包含文字。

### 5.4 required / any：至少一个，不是全部

绑定三个工具，不表示模型必须把三个工具各调用一遍；
想同时查询天气和新闻，仍要观察它返回的所有 `tool_calls`。

**不要对所有问题都强制工具。** 例如用户只说「你好啊」，却只提供天气工具，
模型可能提出不合适的天气查询，甚至补出用户没提供的城市。
强制调用不能代替对用户意图和参数的检查。

`"any"` 是 LangChain 适配层提供的写法；本例会把它转换成 `"required"`，
不代表每家原生 API 都接受字符串 `"any"`。

In [4]:
required_bound = offline_model.bind_tools(tools, tool_choice="required")
any_bound = offline_model.bind_tools(tools, tool_choice="any")

print("required：", required_bound.kwargs["tool_choice"])
print("any：", any_bound.kwargs["tool_choice"])
print("工具数量：", len(required_bound.kwargs["tools"]))
# 这些都只是绑定配置，没有执行天气或新闻函数

required： required
any： required
工具数量： 3


### 5.5 强制选择指定工具

#### 5.5.1 名称字符串和字典写法

沿用课件的 `get_weather1 / get_weather2`，两者用途相似，便于观察强制选择。
正常业务中应让不同工具的职责、描述更清楚。

In [5]:
named_bound = offline_model.bind_tools(
    [get_weather1, get_weather2],
    tool_choice="get_weather2",
)

dict_bound = offline_model.bind_tools(
    [get_weather1, get_weather2],
    tool_choice={
        "type": "function",
        "function": {"name": "get_weather2"},
    },
)

rprint(named_bound.kwargs["tool_choice"])
print("转换后相同：", named_bound.kwargs["tool_choice"] == dict_bound.kwargs["tool_choice"])

{'type': 'function', 'function': {'name': 'get_weather2'}}

转换后相同： True


#### 5.5.2 注意使用工具名，不一定是 Python 变量名

对应你在参数示例中的 `@tool("get_weather_if_forecast")`：
Python 里变量叫 `get_weather`，模型看到的名称却是 `get_weather_if_forecast`。
使用工具对象的 `.name` 可以避免写错。

In [6]:
@tool("get_weather_if_forecast")
def get_weather(city: str, if_forecast: bool = False) -> str:
    """查询当日天气，可以包含明日天气预报。"""
    result = f"{city}今天天气不错"
    if if_forecast:
        result += "，明天天气也不赖"
    return result

renamed_bound = offline_model.bind_tools(
    [get_weather],
    tool_choice=get_weather.name,
)
print("工具名称：", get_weather.name)
rprint(renamed_bound.kwargs["tool_choice"])

工具名称： get_weather_if_forecast


{'type': 'function', 'function': {'name': 'get_weather_if_forecast'}}

#### 5.5.3 名称拼错，不一定在 bind_tools 时就报错

当前适配器可能把不认识的字符串直接留在请求配置中。
「绑定成功」不能证明工具名正确，更不能证明接口调用一定成功。

In [7]:
wrong_bound = offline_model.bind_tools(
    [get_weather1, get_weather2],
    tool_choice="get_weahter2",  # 故意拼错，只检查本地绑定
)
print("本地保存的值：", wrong_bound.kwargs["tool_choice"])
print("真正绑定的名称：", [item.name for item in [get_weather1, get_weather2]])
# 不向真实接口发送这个错误配置

本地保存的值： get_weahter2
真正绑定的名称： ['get_weather1', 'get_weather2']


### 5.6 你的百炼配置：兼容 OpenAI 协议，不代表所有参数都支持

你的现有代码使用 `qwen3.8-max`、`BAILIAN_API_KEY` 和 `BAILIAN_API_URL`。
本笔记保留这些配置，不把其他服务商的支持情况直接套到百炼。

截至整理时（2026-10-16），百炼官方文档说明：

- **Qwen 系列暂不支持 `tool_choice="required"`**；本适配器会把 `any`、`True` 转成 `required`，同样需要注意。
- **思考模式暂不支持强制指定工具**；演示指定工具时，应按模型文档关闭思考。
- 某些模型支持并行工具调用，但必须核对具体模型与接口的支持范围。

关闭思考不会让 `required` 自动获得支持。
当前模型 ID 是否可用、账号是否有权限，仍需要真实接口验证；本次整理不调用模型。

依据：[百炼 Function Calling](https://help.aliyun.com/zh/model-studio/qwen-function-calling)、
[百炼 API 参数说明](https://help.aliyun.com/zh/model-studio/developer-reference/qwen-api)。

### 5.7 工具选择和工具数量要分开看

`required` 的核心含义是「至少一个」，不保证「恰好一个」。
对于支持该参数的服务端，`parallel_tool_calls=False` 可以把**单次响应**限制为零个或一个调用；
再结合受支持的 `required`，才可要求本轮一个。

这不限制整个多轮对话的总调用次数，也不会自动把你的 Python `for` 循环变成并发执行。
下面只观察绑定配置，不向百炼发送这个组合。

In [8]:
single_call_bound = offline_model.bind_tools(
    tools,
    tool_choice="required",
    parallel_tool_calls=False,
)
print("tool_choice：", single_call_bound.kwargs["tool_choice"])
print("parallel_tool_calls：", single_call_bound.kwargs["parallel_tool_calls"])

tool_choice： required
parallel_tool_calls： False


### 5.8 强制选择不等于参数一定合法

`tool_choice` 管选择；`args_schema` 管参数结构和执行前校验。
需要限制输入时，继续使用第三节的 Pydantic 写法。
即使类型和枚举合法，也不能据此证明城市、权限或真实数据符合业务要求。

In [9]:
class WeatherInput(BaseModel):
    city: str = Field(min_length=1, description="城市名称")
    unit: Literal["celsius", "fahrenheit"] = "celsius"

@tool(args_schema=WeatherInput)
def checked_weather(city: str, unit: str = "celsius") -> str:
    """查询天气，执行前校验城市和温度单位。"""
    return f"{city}，单位 {unit}（模拟数据）"

print(checked_weather.invoke({"city": "杭州"}))

for args in [{"city": ""}, {"city": "杭州", "unit": "kelvin"}]:
    try:
        checked_weather.invoke(args)
    except ValidationError as e:
        print(type(e).__name__, e.errors()[0]["msg"])

杭州，单位 celsius（模拟数据）
ValidationError String should have at least 1 character
ValidationError Input should be 'celsius' or 'fahrenheit'


### 5.9 工具结果回来后，不要继续强制同一个工具

`bind_tools(..., tool_choice=...)` 返回带绑定参数的新对象。
如果一直使用绑定了 `required` 或指定工具的对象，再次 `invoke()` 仍会带上这个限制，
可能继续要求调用工具，得不到预期的最终文字回答。

确定工具已经执行完，只需要总结时，可以使用原始 `model.invoke(messages)`，
或重新从原始模型绑定 `tool_choice="none"`。
如果业务确实需要继续查工具，可以改用 `auto` 并继续处理返回的调用。

In [10]:
call_model = offline_model.bind_tools(tools, tool_choice=get_weather2.name)
finish_model = offline_model.bind_tools(tools, tool_choice="none")

print("查询阶段：", call_model.kwargs["tool_choice"])
print("总结阶段：", finish_model.kwargs["tool_choice"])

查询阶段： {'type': 'function', 'function': {'name': 'get_weather2'}}
总结阶段： none


### 5.10 多工具时：每一个 ToolCall 都要有对应结果

下面手动构造 `AIMessage`，离线模拟一次天气和新闻查询。
保留这条 AIMessage，并为每个调用补上匹配编号的 ToolMessage。

对应你的多工具示例：**`messages.append(tool_msg)` 放在整个 `if / elif / else` 之后，仍在 `for` 内部。**
如果只写进 `get_news` 分支，天气结果就会漏掉。

In [11]:
messages = [HumanMessage(content="今天杭州天气如何？今天新闻是什么？")]
response = AIMessage(
    content="",
    tool_calls=[
        {"name": "get_weather1", "args": {"city": "杭州"}, "id": "call_weather", "type": "tool_call"},
        {"name": "get_news", "args": {}, "id": "call_news", "type": "tool_call"},
    ],
)
messages.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather1":
        tool_msg = get_weather1.invoke(tool_call)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
    else:
        raise ValueError(f"不存在的工具：{tool_call['name']}")

    messages.append(tool_msg)  # 每个分支得到的结果都要添加

for message in messages:
    message.pretty_print()

print("调用编号：", [item["id"] for item in response.tool_calls])
print("结果编号：", [msg.tool_call_id for msg in messages if isinstance(msg, ToolMessage)])

================================ Human Message =================================

今天杭州天气如何？今天新闻是什么？
================================== Ai Message ==================================
Tool Calls:
  get_weather1 (call_weather)
 Call ID: call_weather
  Args:
    city: 杭州
  get_news (call_news)
 Call ID: call_news
  Args:
================================= Tool Message =================================
Name: get_weather1

杭州当天晴朗（模拟来源1）
================================= Tool Message =================================
Name: get_news

模拟新闻：这是一条用于演示工具调用的示例消息。
调用编号： ['call_weather', 'call_news']
结果编号： ['call_weather', 'call_news']


### 6、课件实践经验中，另外几个容易误解的地方

#### 6.1 工具失败要明确返回失败，不能伪装成查询成功

课件的除法例子可以返回明确的错误说明；也可以抛出 `ToolException`，
由调用层构造失败的 ToolMessage。下面采用后一种写法，保留调用编号。
`status="error"` 供应用识别，`content` 中也要写清失败原因。

In [12]:
@tool
def divide(a: float, b: float) -> str:
    """计算 a 除以 b。"""
    if b == 0:
        raise ToolException("除数不能为零")
    return f"{a} / {b} = {a / b}"

print(divide.invoke({"a": 6, "b": 2}))

failed_call = {
    "name": divide.name,
    "args": {"a": 6, "b": 0},
    "id": "call_divide_error",
    "type": "tool_call",
}
try:
    divide.invoke(failed_call)
except ToolException as e:
    failed_message = ToolMessage(
        content=f"工具执行失败：{e}",
        tool_call_id=failed_call["id"],
        name=failed_call["name"],
        status="error",
    )
    failed_message.pretty_print()

6.0 / 2.0 = 3.0
================================= Tool Message =================================
Name: divide

工具执行失败：除数不能为零


#### 6.2 不是所有错误都该重试

参数缺失、非法枚举、除零，应先修正参数；重复调用同样的参数不能解决问题。
临时网络错误是否重试、重试几次，应按接口和业务决定。
有写入操作的工具还要确认重复执行的后果。

课件「如果工具失败，尝试其他方法」的提示词只是给模型的行为指导，
不等于程序会自动捕获异常并重试。
原样使用 `@retry(stop=stop_after_attempt(3))` 时，默认耗尽后通常抛出 `RetryError`；
需要重抛最后一次原始异常时，应了解 `reraise=True` 的区别。
本笔记不默认给工具添加重试。

#### 6.3 工具可以返回 dict，不是只能返回 str

对当前工程版本，传参数字典直接 `.invoke()` 可以拿到 Python 字典；
传完整 ToolCall 时，结果会被包装成 ToolMessage，其 `content` 会序列化成文本。

想明确控制输出格式，可以自己 `json.dumps(..., ensure_ascii=False)`。
`\uXXXX` 是合法 JSON 转义，本身不是乱码，也不能仅凭是否转义就断言模型效果好坏。

In [13]:
@tool
def get_user_info(user_id: str) -> dict:
    """获取模拟用户信息。"""
    return {"id": user_id, "name": "张三"}

dict_result = get_user_info.invoke({"user_id": "001"})
user_message = get_user_info.invoke({
    "name": get_user_info.name,
    "args": {"user_id": "001"},
    "id": "call_user_01",
    "type": "tool_call",
})

print("直接调用：", type(dict_result).__name__, dict_result)
print("完整调用：", type(user_message).__name__, user_message.content)
print("显式转 JSON：", json.dumps(dict_result, ensure_ascii=False))

直接调用： dict {'id': '001', 'name': '张三'}
完整调用： ToolMessage {"id": "001", "name": "张三"}
显式转 JSON： {"id": "001", "name": "张三"}


#### 6.4 异步工具使用 await + ainvoke

`async def` 定义的纯异步工具，使用 `.ainvoke()`。
Notebook 已有事件循环，可以直接写 `await`，不必再套 `asyncio.run()`。
只有真正使用异步 I/O，才会获得相应的并发收益；单纯加 `async` 不会让阻塞代码自动变快。

In [14]:
@tool
async def async_weather(city: str) -> str:
    """演示异步工具，返回模拟天气。"""
    return f"{city}天气晴朗（异步模拟）"

async_result = await async_weather.ainvoke({"city": "北京"})
print(async_result)

北京天气晴朗（异步模拟）


### 7、串起来：指定工具查询，再生成最终回答（需要 API）

前面的示例都没有请求真实模型。下面两格沿用你的百炼配置；
为演示指定工具，使用 `extra_body={"enable_thinking": False}` 关闭思考。
具体模型是否支持该字段、模型 ID 是否可用，需要对照实际接入方式验证。
天气函数仍然返回模拟数据。

**本例只演示一次工具往返。** 指定工具获取结果后，使用 `none` 生成总结，
避免继续强制调用天气工具。`required` 的本地示例不直接复制到当前百炼请求。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=BAILIAN_API_KEY,
    base_url=BAILIAN_API_URL,
    extra_body={"enable_thinking": False},
)

In [ ]:
tools = [get_weather1, get_weather2]
tools_by_name = {item.name: item for item in tools}
messages = [HumanMessage(content="请查询今天杭州天气。")]

# 第一轮：明确选择 get_weather2
model_with_tools = model.bind_tools(tools, tool_choice=get_weather2.name)
response = model_with_tools.invoke(messages)
messages.append(response)

if not response.tool_calls:
    raise RuntimeError("接口未返回工具调用，请核对模型、tool_choice 和思考模式。")

for tool_call in response.tool_calls:
    if tool_call["name"] != get_weather2.name:
        raise RuntimeError(f"接口未遵循指定工具：{tool_call['name']}")
    selected_tool = tools_by_name[tool_call["name"]]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

# 第二轮：已有工具结果，只生成最终回答
final_model = model.bind_tools(tools, tool_choice="none")
final_response = final_model.invoke(messages)
messages.append(final_response)

for message in messages:
    message.pretty_print()

### 8、回看时记住这几句

| 容易混淆 | 应该怎么理解 |
| --- | --- |
| `none` 和 `False / None` | 前者明确禁止调用，后两者在本适配器中不显式指定 |
| `required` 和「全部工具」 | 至少一个，不是所有工具各执行一次 |
| `bind_tools` 成功和 API 支持 | 前者是本地配置，后者要核对服务商与模型 |
| 强制调用和参数正确 | 选择工具后，仍要校验入参 |
| 请求调用和执行完成 | AIMessage 是请求，ToolMessage 才承载执行结果 |
| 一直强制和最终总结 | 总结阶段根据业务改用 none、原始模型或 auto |
| 多工具返回 | 每个调用都添加对应的 ToolMessage，编号要匹配 |

参考资料（整理日期：2026-10-16）：

- 课件第 5、6 节：PDF 第 35–44 页。
- [LangChain 工具调用](https://docs.langchain.com/oss/python/langchain/models#tool-calling)
- [百炼 Function Calling](https://help.aliyun.com/zh/model-studio/qwen-function-calling)
- [百炼 API 参数](https://help.aliyun.com/zh/model-studio/developer-reference/qwen-api)
- [Tenacity 异常处理](https://tenacity.readthedocs.io/en/latest/#error-handling)

本地行为以本工程 `langchain-core==1.2.18`、`langchain-openai==1.1.11` 的源码和运行结果为准，
不把未执行的模型请求写成已验证结果。